# Project Setup

This notebook walks you through the one-time setup needed to use this project.
After completing these steps, you can use `cluster.py` from any notebook without
any further configuration.

**Steps:**
1. Install Miniconda (if not already installed)
2. Create the conda environment
3. Write `.cluster_config`
4. Verify the setup

## 1. Install Miniconda

If you are on **Snellius**, Miniconda is already available as a module — skip the download.
Just load it once in your `.bashrc` (edit with `nano ~/.bashrc`) so it is available every session:

```bash
module load 2025
module load Miniconda3/25.5.1-1
source /sw/arch/RHEL9/EB_production/2025/software/Miniconda3/25.5.1-1/etc/profile.d/conda.sh
```

Follow the prompts, then restart your terminal or run `source ~/.bashrc`.

## 2. Create the conda environment

Run the following in your terminal (not in this notebook).
This creates a conda environment with all required packages.

```bash
conda create -n diffusion python=3.11 -y
conda activate diffusion
pip install -r requirements.txt
```

You can name the environment anything you like — just make sure to use the
same name when writing `.cluster_config` in the next step.

To verify the environment was created correctly:
```bash
conda activate diffusion
python -c "import torch; print(torch.__version__)"
```

To register the environment as a Jupyter kernel (so you can select it in VS Code or Jupyter):
```bash
conda activate diffusion
python -m ipykernel install --user --name diffusion --display-name "diffusion"
```

## 3. Write `.cluster_config`

`cluster.py` reads a file called `.cluster_config` in the project root to know
where the project lives and which conda environment to use. Run the cell below
to write it — just fill in your own paths and environment name first.

In [1]:
import os
import platform

# --- fill these in ---
PROJECT_DIR = '/home/scur0036/diffusion-models-project'   # absolute path to project root
CONDA_ENV   = 'diffusion'                                  # name of your conda environment
# ---------------------

ON_SNELLIUS = 'snellius' in platform.node().lower()

config_path = os.path.join(PROJECT_DIR, '.cluster_config')
with open(config_path, 'w') as f:
    f.write(f'PROJECT_DIR={PROJECT_DIR}\n')
    f.write(f'CONDA_ENV={CONDA_ENV}\n')
    f.write(f'ON_SNELLIUS={ON_SNELLIUS}\n')

print(f'Written to {config_path}')
print()
print(open(config_path).read())

Written to /home/scur0036/diffusion-models-project/.cluster_config

PROJECT_DIR=/home/scur0036/diffusion-models-project
CONDA_ENV=diffusion
ON_SNELLIUS=True



`.cluster_config` is machine-specific and is listed in `.gitignore` — it will not
be committed to the repository. Every person who clones the project runs this
cell once with their own paths.

If you ever move the project folder or rename the conda environment, just rerun
the cell above with the updated values.

## 4. Verify the setup

Run the cells below to confirm that `cluster.py` picks up the config correctly
and that all imports work.

In [2]:
import cluster

print(f'PROJECT_DIR : {cluster.PROJECT_DIR}')
print(f'CONDA_ENV   : {cluster.CONDA_ENV}')
print(f'MODELS_DIR  : {cluster.MODELS_DIR}')
print(f'LOGS_DIR    : {cluster.LOGS_DIR}')
print(f'ON_SNELLIUS : {cluster.ON_SNELLIUS}')
print(f'ENV_TYPE    : {cluster.ENV_TYPE}')

PROJECT_DIR : /home/scur0036/diffusion-models-project
CONDA_ENV   : diffusion
MODELS_DIR  : /home/scur0036/diffusion-models-project/models
LOGS_DIR    : /home/scur0036/diffusion-models-project/logs
ON_SNELLIUS : True
ENV_TYPE    : conda


In [ ]:
import torch
from ddpm import NoiseScheduler, UNet
from ddpm.dataset import load_mnist

print(f'torch version : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print('All imports OK.')

## Done

You are ready to use the project. See `example_usage.py` for a walkthrough of
the full training and generation workflow, and `experiment_workflow.py` for
how to run and compare multiple models.

To submit jobs to the cluster from any notebook, import `cluster` and use:
- `cluster.train_on_cluster(...)` — training only
- `cluster.train_and_generate_on_cluster(...)` — training + image generation
- `cluster.generate_on_cluster(...)` — generation from an already trained model
- `cluster.job_status(job_id)` — check job status
- `cluster.cancel_job(job_id)` — cancel a job